# Cắt bỏ thành phần mô hình trên NF-UNSW-NB15-v3

Notebook đánh giá bốn cấu hình bằng cách huấn luyện lại từ đầu cả hai giai đoạn trên cùng tập dữ liệu và phép chia của NB2. Mốc đối chiếu cũng được chạy trong phiên này. Điểm so sánh chính là Macro F1 đa lớp trên tập kiểm thử, luôn đọc cùng recall của lớp tấn công và mốc F1 của bộ phân loại luôn dự đoán tấn công.

Xáo trộn định danh đỉnh kiểm tra vai trò của cấu trúc liên kết quan sát được. Bỏ tám cột IAT chỉ kiểm tra nhóm thuộc tính thời gian giữa các gói. Đặt số tầng chú ý đồ thị bằng không giữ bộ nhớ TGN và các đặc trưng cạnh đi thẳng vào đầu phân loại, nhưng bỏ bước tổng hợp lân cận. Vì vậy ba phép đối chứng trả lời ba câu hỏi khác nhau.

## 1. Môi trường và mã nguồn

Sử dụng cùng gói mã nguồn với NB2. Một bản mã duy nhất trong dữ liệu đầu vào và GPU tương thích là điều kiện để bắt đầu huấn luyện.

In [1]:
!pip install -q torch_geometric

import glob
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

if not torch.cuda.is_available():
    raise RuntimeError('Cần bật GPU cho thí nghiệm huấn luyện lại.')
capability = torch.cuda.get_device_capability(0)
supported = torch.cuda.get_arch_list()
if f'sm_{capability[0]}{capability[1]}' not in supported:
    raise RuntimeError(f'PyTorch không hỗ trợ GPU {torch.cuda.get_device_name(0)}: {supported}')

source_roots = [Path(p) for p in glob.glob('/kaggle/input/**/src', recursive=True)
                if (Path(p) / 'tgn_nids' / 'experiments' / 'runner.py').is_file()]
if len(source_roots) != 1:
    raise RuntimeError(f'Cần đúng một gói mã nguồn, tìm được {len(source_roots)}: {source_roots}')
sys.path.insert(0, str(source_roots[0]))
import tgn_nids
from tgn_nids.data.preprocess import NetFlowPreprocessor
from tgn_nids.experiments.runner import ExperimentConfig, run_experiment
from tgn_nids.models.embedding import TemporalEmbeddingModule
from tgn_nids.utils import figures as fig_lib
fig_lib.apply_style()
print(f'GPU: {torch.cuda.get_device_name(0)} | Mã nguồn: {tgn_nids.SOURCE_VERSION}')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 42.0 MB/s eta 0:00:00
GPU: Tesla T4 | Mã nguồn: 1.1.0


## 2. Dữ liệu và phép chia

Giữ toàn bộ NF-UNSW-NB15-v3. Sắp tăng dần theo `FLOW_START_MILLISECONDS` bằng thuật toán ổn định rồi lấy 80% đầu để huấn luyện, 10% tiếp theo để kiểm định và 10% cuối để kiểm thử. Kiểm sự hiện diện của mọi loại tấn công ở cả ba tập trước khi chạy. Bộ tiền xử lý chỉ khớp trên tập huấn luyện của từng giai đoạn.

In [2]:
NOTEBOOK_ID = 'twoDTS_ablation'
OUTPUT_ROOT = Path('/kaggle/working/outputs') / NOTEBOOK_ID
SUBFOLDERS = ('models', 'metrics', 'tables', 'figures')
SEED = 42

def cell_dir(name, kind):
    if kind not in SUBFOLDERS:
        raise ValueError(kind)
    path = OUTPUT_ROOT / name / kind
    path.mkdir(parents=True, exist_ok=True)
    return path

candidates = [Path(p) for p in glob.glob('/kaggle/input/**/*.parquet', recursive=True)
              if all(s in Path(p).name.lower() for s in ('unsw', 'nb15', 'v3'))]
if len(candidates) != 1:
    raise RuntimeError(f'Cần đúng một tệp UNSW-v3, tìm được: {candidates}')
frame = pd.read_parquet(candidates[0])
frame.columns = frame.columns.str.strip()
required = {'FLOW_START_MILLISECONDS', 'Label', 'Attack', 'IPV4_SRC_ADDR', 'IPV4_DST_ADDR'}
if not required.issubset(frame.columns):
    raise ValueError(f'Thiếu cột bắt buộc: {sorted(required - set(frame.columns))}')
if len(frame) != 2365424:
    raise ValueError(f'Khối lượng dữ liệu khác NB2: {len(frame):,}')
ordered = frame.sort_values('FLOW_START_MILLISECONDS', kind='mergesort').reset_index(drop=True)
train_end = int(len(ordered) * 0.80)
val_end = int(len(ordered) * 0.90)
splits = {'train': ordered.iloc[:train_end].copy(),
          'val': ordered.iloc[train_end:val_end].copy(),
          'test': ordered.iloc[val_end:].copy()}
classes = set(frame['Attack'].astype(str).unique())
for name, part in splits.items():
    if set(part['Attack'].astype(str).unique()) != classes:
        raise RuntimeError(f'Tập {name} thiếu loại tấn công.')
    if not np.all(np.diff(part['FLOW_START_MILLISECONDS'].to_numpy()) >= 0):
        raise RuntimeError(f'Tập {name} chưa được sắp theo thời gian.')
if not (splits['train']['FLOW_START_MILLISECONDS'].max() <=
        splits['val']['FLOW_START_MILLISECONDS'].min() <=
        splits['test']['FLOW_START_MILLISECONDS'].min()):
    raise RuntimeError('Các ranh giới thời gian chồng lấn.')
fig_lib.write_manifest(str(cell_dir('01_du_lieu', 'metrics')), {
    'dataset': 'NF-UNSW-NB15-v3', 'source_file': candidates[0].name,
    'rows': len(frame), 'sample_ratio': 1.0, 'split_seed': SEED,
    'split': 'chronological_80_10_10',
    'counts': {name: len(part) for name, part in splits.items()},
    'source_version': tgn_nids.SOURCE_VERSION}, name='data_protocol')
print('NF-UNSW-NB15-v3 | toàn bộ dữ liệu | chia theo mốc thời gian')
for name, part in splits.items():
    print(f'{name:<5} {len(part):>9,} flow | tấn công {int(part.Label.sum()):>7,}')


NF-UNSW-NB15-v3 | toàn bộ dữ liệu | chia theo mốc thời gian
train 1,892,339 flow | tấn công  86,301
val     236,542 flow | tấn công  21,296
test    236,543 flow | tấn công  20,096


## 3. Cấu hình và kiểm tra trước khi chạy

Giữ kích thước biểu diễn, hạt giống, ngân sách epoch, patience, tiêu chí dừng sớm và quy tắc ngưỡng như NB2. Mốc đối chiếu được huấn luyện lại trong cùng môi trường. Trường hợp không tổng hợp lân cận dùng `num_layers=0`, vì hai vector bộ nhớ và biểu diễn đều có 192 chiều. Khi bỏ IAT, cả bảng nguồn lẫn ba tập chia phải nhất quán và số chiều cạnh phải giảm từ 49 xuống 41.

In [3]:
IAT_COLUMNS = (
    'SRC_TO_DST_IAT_MIN', 'SRC_TO_DST_IAT_MAX',
    'SRC_TO_DST_IAT_AVG', 'SRC_TO_DST_IAT_STDDEV',
    'DST_TO_SRC_IAT_MIN', 'DST_TO_SRC_IAT_MAX',
    'DST_TO_SRC_IAT_AVG', 'DST_TO_SRC_IAT_STDDEV',
)
if not set(IAT_COLUMNS).issubset(frame.columns):
    raise ValueError(f'Thiếu cột IAT: {sorted(set(IAT_COLUMNS) - set(frame.columns))}')
if len(NetFlowPreprocessor()._get_feature_cols(splits['train'])) != 49:
    raise RuntimeError('Số đặc trưng gốc không bằng 49.')
probe = TemporalEmbeddingModule(192, 192, 49, num_heads=8, num_layers=0)
assert len(probe.gat_layers) == 0 and len(probe.norms) == 0
assert probe(torch.zeros(2, 192), torch.tensor([[0], [1]]),
             torch.zeros(1, 49)).shape == (2, 192)
del probe

VARIANTS = {
    'baseline': {'node_identity': False, 'features': (), 'num_layers': 1},
    'random_nodes': {'node_identity': True, 'features': (), 'num_layers': 1},
    'no_iat': {'node_identity': False, 'features': IAT_COLUMNS, 'num_layers': 1},
    'no_propagation': {'node_identity': False, 'features': (), 'num_layers': 0},
}
results = {}
print('Tiền kiểm đạt: 49 đặc trưng, 8 cột IAT, tầng chú ý có thể tắt.')


Tiền kiểm đạt: 49 đặc trưng, 8 cột IAT, tầng chú ý có thể tắt.


## 4. Huấn luyện từng cấu hình

Mỗi cấu hình bắt đầu từ trọng số khởi tạo với seed 42. Giai đoạn một chỉ học từ luồng lành tính ở tập huấn luyện, chọn checkpoint theo PR-AUC kiểm định và hiệu chỉnh ngưỡng bằng Macro F1 trên tập kiểm định. Giai đoạn hai nạp bộ mã hóa của chính cấu hình đó, giữ cố định trọng số bộ mã hóa nhưng tiếp tục cập nhật trạng thái bộ nhớ, rồi học đầu phân loại đa lớp. Macro F1 kiểm định chọn checkpoint giai đoạn hai. Mỗi cấu hình ghi checkpoint, kết quả máy đọc, bảng và hình trong thư mục riêng của ô gọi.

In [4]:
def run_variant(key, cell_name):
    spec = VARIANTS[key]
    removed = tuple(spec['features'])
    # Tập chia truyền vào runner phải mất cùng các cột mà cấu hình khai báo.
    parts = {name: part.drop(columns=list(removed)).copy()
             for name, part in splits.items()}
    expected_dim = 49 - len(removed)
    assert len(NetFlowPreprocessor()._get_feature_cols(parts['train'])) == expected_dim
    for part in parts.values():
        assert not set(removed).intersection(part.columns)
    base = dict(protocol='chronological_80_10_10', memory_dim=192,
                embedding_dim=192, time_dim=64, hidden_dim=192,
                num_heads=8, num_layers=spec['num_layers'], dropout=0.1,
                classifier_dropout=0.3, batch_size=512,
                learning_rate=1e-4, weight_decay=1e-5,
                log_every_epochs=10, seed=SEED, sample_ratio=1.0,
                ablate_node_identity=spec['node_identity'],
                ablate_features=removed)
    benign = parts['train'][parts['train']['Label'].astype(int) == 0].copy()
    first_parts = {'train': benign, 'val': parts['val'], 'test': parts['test']}
    first_name = f'stage1_unsw_v3_{key}'
    first = ExperimentConfig(name=first_name, description=f'Tự giám sát: {key}',
                             task='binary', mode='self_supervised', mask_ratio=0.15,
                             num_epochs=150, early_stopping_patience=25,
                             early_stopping_metric='pr_auc',
                             threshold_strategy='f1', **base)
    model_dir = cell_dir(cell_name, 'models')
    print(f'Bắt đầu {key}: giai đoạn một, {len(benign):,} luồng lành tính')
    stage1 = run_experiment(frame, first, NetFlowPreprocessor(),
                            str(model_dir), splits=first_parts, verbose=True)
    checkpoint = model_dir / f'model_{first_name}.pt'
    second_name = f'stage2_unsw_v3_{key}'
    second = ExperimentConfig(name=second_name, description=f'Đa lớp: {key}',
                              task='multiclass', mode='supervised',
                              num_epochs=100, early_stopping_patience=15,
                              early_stopping_metric='f1_macro',
                              pretrained_encoder=str(checkpoint),
                              freeze_encoder=True, **base)
    print(f'Bắt đầu {key}: giai đoạn hai')
    stage2 = run_experiment(frame, second, NetFlowPreprocessor(),
                            str(model_dir), splits=parts, verbose=True)
    for stage, expected_name in ((stage1, first_name), (stage2, second_name)):
        stored = json.loads((model_dir / f'result_{expected_name}.json').read_text(encoding='utf-8'))
        assert stored['ablation'] == {'node_identity': spec['node_identity'],
                                      'features': list(removed)}
        assert stored['edge_dim'] == expected_dim
        assert stored['config']['num_layers'] == spec['num_layers']
        assert stored['split_stats']['test']['count'] == len(parts['test'])
        assert stored['best_epoch'] is not None
    binary = stage2['metrics']['binary_equivalent']
    attack_rate = binary['support_attack'] / len(parts['test'])
    row = {'variant': key, 'dataset': 'NF-UNSW-NB15-v3',
           'sample_ratio': 1.0, 'question': 'retrain',
           'random_nodes': spec['node_identity'], 'removed_features': len(removed),
           'num_layers': spec['num_layers'], 'edge_dim': expected_dim,
           'f1_macro': stage2['metrics']['f1_macro'],
           'attack_recall': binary['recall'], 'attack_f1': binary['f1'],
           'far': binary['far'], 'trivial_f1': fig_lib.trivial_f1(attack_rate),
           'stage1_best_epoch': stage1['best_epoch'],
           'stage2_best_epoch': stage2['best_epoch'],
           'stage1_epochs_run': len(stage1['history']),
           'stage2_epochs_run': len(stage2['history']),
           'training_time_seconds': stage1['training_time_seconds'] + stage2['training_time_seconds']}
    fig_lib.export_table([row], str(cell_dir(cell_name, 'tables')), 'metrics')
    fig_lib.write_manifest(str(cell_dir(cell_name, 'metrics')),
                           {'variant': key, 'settings': spec, 'summary': row,
                            'stage1_result': f'result_{first_name}.json',
                            'stage2_result': f'result_{second_name}.json'},
                           name='ablation_protocol')
    fig_lib.plot_confusion_matrix(stage2['metrics']['confusion_matrix'],
                                  stage2['class_names'], str(cell_dir(cell_name, 'figures')),
                                  name='confusion_normalized', normalize=True)
    plt.close('all')
    fig_lib.plot_per_class(stage2['metrics']['per_class'],
                           str(cell_dir(cell_name, 'figures')), name='per_class')
    plt.close('all')
    results[key] = row
    print(f"{key}: Macro F1 {row['f1_macro'] * 100:.3f}% | "
          f"Recall tấn công {row['attack_recall'] * 100:.3f}% | "
          f"Mốc F1 {row['trivial_f1'] * 100:.3f}%")
    return row


In [5]:
baseline = run_variant('baseline', '04_baseline')


Bắt đầu baseline: giai đoạn một, 1,806,038 luồng lành tính

------
Thí nghiệm: stage1_unsw_v3_baseline
Mô tả: Tự giám sát: baseline
Protocol: chronological_80_10_10 | Task: binary | Mode: self_supervised | Device: cuda
------
Dùng tập đã phân chia sẵn do phía gọi cung cấp, bỏ qua giao thức 'chronological_80_10_10'
Train:   1,806,038 flow |        0 attack
Val:       236,542 flow |   21,296 attack
Test:      236,543 flow |   20,096 attack
------
Danh mục nhãn: 10 lớp | Benign ở vị trí 2
Edge features: 49 | Số node: 44
Masked reconstruction: che 7/49 thuộc tính mỗi cạnh (mask_ratio 0.15)
Tham số: 1,539,441
------
Epoch   1 | Train loss:    0.31436 | Val loss:   10.48963 | Val pr_auc:   0.96462
Epoch  10 | Train loss:    0.09675 | Val loss:    3.62432 | Val pr_auc:   0.98615
Epoch  20 | Train loss:    0.07484 | Val loss:    1.66578 | Val pr_auc:   0.98754
Epoch  30 | Train loss:    0.08435 | Val loss:    1.40004 | Val pr_auc:   0.98634
Early stopping tại epoch 37
------
Nạp lại trọng số t

In [6]:
random_nodes = run_variant('random_nodes', '05_random_nodes')


Bắt đầu random_nodes: giai đoạn một, 1,806,038 luồng lành tính

------
Thí nghiệm: stage1_unsw_v3_random_nodes
Mô tả: Tự giám sát: random_nodes
Protocol: chronological_80_10_10 | Task: binary | Mode: self_supervised | Device: cuda
------
Dùng tập đã phân chia sẵn do phía gọi cung cấp, bỏ qua giao thức 'chronological_80_10_10'
Train:   1,806,038 flow |        0 attack
Val:       236,542 flow |   21,296 attack
Test:      236,543 flow |   20,096 attack
------
Danh mục nhãn: 10 lớp | Benign ở vị trí 2
Ablation, đã gán lại node nguồn và node đích một cách ngẫu nhiên
Edge features: 49 | Số node: 44
Masked reconstruction: che 7/49 thuộc tính mỗi cạnh (mask_ratio 0.15)
Tham số: 1,539,441
------
Epoch   1 | Train loss:    0.31842 | Val loss:    8.64240 | Val pr_auc:   0.98140
Epoch  10 | Train loss:    0.09384 | Val loss:    1.80084 | Val pr_auc:   0.97274
Epoch  20 | Train loss:    0.06795 | Val loss:    1.65212 | Val pr_auc:   0.98146
Epoch  30 | Train loss:    0.07473 | Val loss:    1.57399 

In [7]:
no_iat = run_variant('no_iat', '06_no_iat')


Bắt đầu no_iat: giai đoạn một, 1,806,038 luồng lành tính

------
Thí nghiệm: stage1_unsw_v3_no_iat
Mô tả: Tự giám sát: no_iat
Protocol: chronological_80_10_10 | Task: binary | Mode: self_supervised | Device: cuda
------
Ablation, đã loại khỏi dữ liệu: SRC_TO_DST_IAT_MIN, SRC_TO_DST_IAT_MAX, SRC_TO_DST_IAT_AVG, SRC_TO_DST_IAT_STDDEV, DST_TO_SRC_IAT_MIN, DST_TO_SRC_IAT_MAX, DST_TO_SRC_IAT_AVG, DST_TO_SRC_IAT_STDDEV
Dùng tập đã phân chia sẵn do phía gọi cung cấp, bỏ qua giao thức 'chronological_80_10_10'
Train:   1,806,038 flow |        0 attack
Val:       236,542 flow |   21,296 attack
Test:      236,543 flow |   20,096 attack
------
Danh mục nhãn: 10 lớp | Benign ở vị trí 2
Edge features: 41 | Số node: 44
Masked reconstruction: che 6/41 thuộc tính mỗi cạnh (mask_ratio 0.15)
Tham số: 1,517,929
------
Epoch   1 | Train loss:    0.28133 | Val loss:   12.88024 | Val pr_auc:   0.94817
Epoch  10 | Train loss:    0.06739 | Val loss:    2.51198 | Val pr_auc:   0.98888
Epoch  20 | Train loss:   

In [8]:
no_propagation = run_variant('no_propagation', '07_no_propagation')


Bắt đầu no_propagation: giai đoạn một, 1,806,038 luồng lành tính

------
Thí nghiệm: stage1_unsw_v3_no_propagation
Mô tả: Tự giám sát: no_propagation
Protocol: chronological_80_10_10 | Task: binary | Mode: self_supervised | Device: cuda
------
Dùng tập đã phân chia sẵn do phía gọi cung cấp, bỏ qua giao thức 'chronological_80_10_10'
Train:   1,806,038 flow |        0 attack
Val:       236,542 flow |   21,296 attack
Test:      236,543 flow |   20,096 attack
------
Danh mục nhãn: 10 lớp | Benign ở vị trí 2
Edge features: 49 | Số node: 44
Masked reconstruction: che 7/49 thuộc tính mỗi cạnh (mask_ratio 0.15)
Tham số: 537,393
------
Epoch   1 | Train loss:    0.32214 | Val loss:   10.83459 | Val pr_auc:   0.94275
Epoch  10 | Train loss:    0.10515 | Val loss:    3.28429 | Val pr_auc:   0.97462
Epoch  20 | Train loss:    0.08040 | Val loss:    1.79630 | Val pr_auc:   0.97104
Epoch  30 | Train loss:    0.07703 | Val loss:    1.59213 | Val pr_auc:   0.97756
Epoch  40 | Train loss:    0.08578 | 

## 5. Đối chiếu và xuất kết quả

Bảng tổng hợp đọc trực tiếp từ kết quả từng cấu hình. Giá trị chênh lệch là phép trừ giữa hai chỉ số đo trong phiên này, không phải một kết luận nhân quả cho mọi dữ liệu khác. Nếu vòng tốt nhất nằm sát trần ngân sách thì kết quả được gắn cờ để xem lại trước khi đưa vào báo cáo.

In [9]:
rows = [results[name] for name in VARIANTS]
for row in rows:
    row['delta_macro_f1_pp'] = (row['f1_macro'] - baseline['f1_macro']) * 100
    row['near_epoch_limit'] = (row['stage1_best_epoch'] > 125 or
                               row['stage2_best_epoch'] > 85)
fig_lib.export_table(rows, str(cell_dir('08_tong_hop', 'tables')), 'ablation_summary')
fig_lib.write_manifest(str(cell_dir('08_tong_hop', 'metrics')),
                       {'dataset': 'NF-UNSW-NB15-v3', 'sample_ratio': 1.0,
                        'source_version': tgn_nids.SOURCE_VERSION,
                        'configurations': list(VARIANTS), 'rows': rows},
                       name='ablation_summary')
fig, ax = plt.subplots(figsize=(6.4, 3.5))
names = list(VARIANTS)
values = [results[name]['f1_macro'] * 100 for name in names]
bars = ax.bar(names, values, color=[fig_lib.ACCENT] + [fig_lib.NEUTRAL] * 3)
ax.bar_label(bars, fmt='%.2f', padding=2)
ax.set_ylabel('Macro F1 (%)')
ax.set_ylim(0, max(values) * 1.18)
ax.tick_params(axis='x', labelrotation=15)
fig.tight_layout()
fig_lib.save_figure(fig, str(cell_dir('08_tong_hop', 'figures')), 'ablation_macro_f1')
plt.close(fig)
print(pd.DataFrame(rows)[['variant', 'f1_macro', 'attack_recall',
                          'trivial_f1', 'delta_macro_f1_pp', 'near_epoch_limit']].to_string(index=False))
if any(row['near_epoch_limit'] for row in rows):
    print('Có cấu hình cần xem lại ngân sách epoch trước khi trích dẫn.')
print(f'Kết quả: {OUTPUT_ROOT}')


       variant  f1_macro  attack_recall  trivial_f1  delta_macro_f1_pp  near_epoch_limit
      baseline  0.549362       0.999801    0.156609           0.000000             False
  random_nodes  0.521089       0.999453    0.156609          -2.827307             False
        no_iat  0.546163       0.999751    0.156609          -0.319915             False
no_propagation  0.544441       0.999801    0.156609          -0.492053             False
Kết quả: /kaggle/working/outputs/twoDTS_ablation
